# Tarea 3: Polars

## 1. Estructura del proyecto

1. Configuración del entorno
2. Dataset
3. Parte 1 — Análisis Exploratorio
4. Parte 2 — Ingeniería de Características
5. Parte 3 — Machine Learning
6. Benchmark Polars vs Pandas
7. Experimentos (escalabilidad y lazy execution)

## 2. Configuración del Entorno


In [ ]:
import sys

!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q kagglehub polars xgboost psutil matplotlib

print("Dependencias instaladas correctamente.")

In [ ]:
# Imports globales del proyecto

import os
import time
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 11

print("Polars:", pl.__version__)

## 3. Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("abderrahimalakouche/flight-delay-prediction")
print("Ruta del dataset:", path)
print("Archivos:")
for archivo in os.listdir(path):
    print("  ", archivo)

In [ ]:
# Carga con Polars

ruta_csv = os.path.join(path, "Data.csv")
df = pl.read_csv(ruta_csv)

print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print()
print(df.schema)

## 4. Parte 1: Análisis Exploratorio

### 4.1 Estadísticas descriptivas y valores faltantes

In [ ]:
print("=== Primeras 5 filas ===")
print(df.head(5))

print("\n=== Estadísticas descriptivas ===")
print(df.describe())

print("\n=== Valores nulos por columna ===")
print(df.null_count())

### 4.2 Distribución de variables

In [ ]:
print("=== Distribución de target (minutos de retraso) ===")
print(df["target"].describe())

print("\n=== Distribución de STATUS ===")
print(df["STATUS"].value_counts().sort("count", descending=True))

In [ ]:
target_vals = df["target"].to_numpy()

fig, ax = plt.subplots(figsize=(11, 4))

ax.hist(target_vals[(target_vals >= 0) & (target_vals <= 300)],
        bins=60, color="#3498db", edgecolor="white")
ax.axvline(15, color="#e74c3c", linestyle="--", linewidth=2,
           label="Umbral IATA = 15 min")
ax.set_title("Distribución del retraso (target) — 0 a 300 min", fontweight="bold")
ax.set_xlabel("Minutos de retraso")
ax.set_ylabel("Cantidad de vuelos")
ax.legend()
plt.tight_layout()
plt.savefig("eda_distribucion_target.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: eda_distribucion_target.png")

### 4.3 Derivación de variables numéricas (temporales)

In [ ]:
def parsear_datetime_mixto(nombre_columna: str) -> pl.Expr:
    return pl.coalesce([
        pl.col(nombre_columna).str.to_datetime(format="%Y-%m-%d %H.%M.%S", strict=False),
        pl.col(nombre_columna).str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False),
    ])

df_con_fechas = df.with_columns([
    pl.col("DATOP").str.to_date(format="%Y-%m-%d").alias("fecha_operacion"),
    parsear_datetime_mixto("STD").alias("datetime_salida"),
    parsear_datetime_mixto("STA").alias("datetime_llegada"),
])

df_con_fechas = df_con_fechas.with_columns([
    pl.col("fecha_operacion").dt.month().alias("mes"),
    pl.col("fecha_operacion").dt.weekday().alias("dia_semana"),
    pl.col("datetime_salida").dt.hour().alias("hora_salida"),
    pl.col("datetime_llegada").dt.hour().alias("hora_llegada_programada"),
])

print("=== Nulls tras el parseo de fechas/horas ===")
print(df_con_fechas.select([
    "datetime_salida", "datetime_llegada",
    "mes", "dia_semana", "hora_salida", "hora_llegada_programada"
]).null_count())

print("\n=== Vista de las nuevas columnas ===")
print(df_con_fechas.select([
    "DATOP", "mes", "dia_semana", "STD", "hora_salida", "STA", "hora_llegada_programada"
]).head(5))

### 4.4 Análisis de correlaciones

In [ ]:
cols_num = ["mes", "dia_semana", "hora_salida", "hora_llegada_programada", "target"]

df_corr = df_con_fechas.select(cols_num).drop_nulls()
matriz_corr = df_corr.corr().to_numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(matriz_corr, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(cols_num)))
ax.set_yticks(range(len(cols_num)))
ax.set_xticklabels(cols_num, rotation=45, ha="right")
ax.set_yticklabels(cols_num)

for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        ax.text(j, i, f"{matriz_corr[i, j]:.2f}", ha="center", va="center",
                color="black", fontsize=10)

ax.set_title("Matriz de Correlación (variables numéricas)", fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("eda_correlacion.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: eda_correlacion.png")

## 5. Parte 2: Ingeniería de Características

### 5.1 Filtrado de registros y variable objetivo

In [ ]:
df_filtrado = df_con_fechas.filter(pl.col("STATUS") == "ATA")
print(f"Filas tras filtrar STATUS == ATA: {df_filtrado.shape[0]:,}")

print("\n=== Cuantiles de target ===")
print(df_filtrado.select([
    pl.col("target").quantile(0.50).alias("mediana"),
    pl.col("target").quantile(0.95).alias("p95"),
    pl.col("target").quantile(0.99).alias("p99"),
    pl.col("target").max().alias("max"),
]))

outliers = df_filtrado.filter(pl.col("target") > 300).shape[0]
print(f"\nRegistros con retraso > 300 min: {outliers:,} "
      f"({outliers/df_filtrado.shape[0]*100:.2f}%)")

df_limpio = df_filtrado.filter(pl.col("target") <= 300)
df_limpio = df_limpio.with_columns([
    (pl.col("target") > 15).cast(pl.Int8).alias("retrasado")
])
print(f"\nFilas tras eliminar outliers: {df_limpio.shape[0]:,}")

conteo = df_limpio["retrasado"].value_counts().sort("retrasado")
total = df_limpio.shape[0]
print("\n=== Balance de clases ===")
for row in conteo.iter_rows(named=True):
    etiqueta = "Retrasado" if row["retrasado"] == 1 else "A tiempo "
    print(f"{etiqueta}: {row['count']:,} vuelos ({row['count']/total*100:.1f}%)")

### 5.2 Balance de clases (gráfica)

In [ ]:
conteo_dict = {row["retrasado"]: row["count"] for row in conteo.iter_rows(named=True)}
etiquetas = ["A tiempo (0)", "Retrasado (1)"]
conteos   = [conteo_dict.get(0, 0), conteo_dict.get(1, 0)]
colores   = ["#2ecc71", "#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(etiquetas, conteos, color=colores, edgecolor="white", linewidth=1.5)
axes[0].set_title("Distribución de la Variable Objetivo")
axes[0].set_ylabel("Cantidad de vuelos")
for i, v in enumerate(conteos):
    axes[0].text(i, v + 300, f"{v:,}", ha="center", fontweight="bold")

axes[1].pie(conteos, labels=etiquetas, colors=colores, autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Proporción de Clases")

plt.suptitle("Balance de Clases — Vuelos Tunisair", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("balance_clases.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: balance_clases.png")

### 5.3 Creación de nuevas características

In [ ]:
df_features = df_limpio.with_columns([
    pl.col("FLTID").str.slice(0, 2).str.strip_chars().alias("codigo_aerolinea")
])

df_features = df_features.with_columns([
    pl.col("AC")
      .str.extract(r"([A-Z]{0,2}\d{2,3}[A-Z]?)", 1)
      .fill_null("desconocido")
      .alias("modelo_aeronave")
])

print("=== Modelos de aeronave extraídos (top 10) ===")
print(df_features["modelo_aeronave"].value_counts().sort("count", descending=True).head(10))

### 5.4 Agregaciones con `group_by`

In [ ]:
retraso_por_aeropuerto = (
    df_features.group_by("DEPSTN")
    .agg([
        pl.col("target").mean().alias("retraso_promedio_aeropuerto"),
        pl.col("target").count().alias("total_vuelos_aeropuerto"),
    ])
)

retraso_por_ruta = (
    df_features.group_by(["DEPSTN", "ARRSTN"])
    .agg([
        pl.col("target").mean().alias("retraso_promedio_ruta"),
        pl.col("target").count().alias("total_vuelos_ruta"),
    ])
)

print("=== Aeropuertos más retrasados (mín. 30 vuelos) ===")
print(retraso_por_aeropuerto
      .filter(pl.col("total_vuelos_aeropuerto") >= 30)
      .sort("retraso_promedio_aeropuerto", descending=True).head(10))

print("\n=== Rutas más retrasadas (mín. 30 vuelos) ===")
print(retraso_por_ruta
      .filter(pl.col("total_vuelos_ruta") >= 30)
      .sort("retraso_promedio_ruta", descending=True).head(10))

### 5.5 Joins y manejo de datos faltantes

In [ ]:
df_con_features = df_features.join(
    retraso_por_aeropuerto.select(["DEPSTN", "retraso_promedio_aeropuerto"]),
    on="DEPSTN", how="left"
)

df_con_features = df_con_features.join(
    retraso_por_ruta.select(["DEPSTN", "ARRSTN", "retraso_promedio_ruta"]),
    on=["DEPSTN", "ARRSTN"], how="left"
)

print("=== Nulls tras los joins ===")
print(df_con_features.select(["retraso_promedio_aeropuerto", "retraso_promedio_ruta"]).null_count())

mediana_global = df_con_features["target"].median()
df_con_features = df_con_features.with_columns([
    pl.col("retraso_promedio_aeropuerto").fill_null(mediana_global),
    pl.col("retraso_promedio_ruta").fill_null(mediana_global),
])

print(f"\nMediana global de retraso: {mediana_global} min")
print("\n=== Nulls tras rellenar ===")
print(df_con_features.select(["retraso_promedio_aeropuerto", "retraso_promedio_ruta"]).null_count())
print(f"\nShape final con features: {df_con_features.shape}")

### 5.6 Dataset final para el modelo (encoding)

In [ ]:
df_modelo = (
    df_con_features
    .select([
        "mes", "dia_semana", "hora_salida", "hora_llegada_programada",
        "retraso_promedio_aeropuerto", "retraso_promedio_ruta",
        "codigo_aerolinea", "modelo_aeronave", "retrasado",
    ])
    .with_columns([
        pl.col("codigo_aerolinea").cast(pl.Categorical).to_physical().alias("codigo_aerolinea"),
        pl.col("modelo_aeronave").cast(pl.Categorical).to_physical().alias("modelo_aeronave"),
    ])
)

print(f"Shape del dataset final: {df_modelo.shape}")
print("\n=== Nulls por columna ===")
print(df_modelo.null_count())
print("\n=== Tipos de datos ===")
for col in df_modelo.columns:
    print(f"  {col}: {df_modelo[col].dtype}")
print("\n=== Primeras 5 filas ===")
print(df_modelo.head(5))

## 6. Parte 3: Machine Learning

### 6.1 Entrenamiento y evaluación

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import xgboost as xgb

# Polars -> numpy para sklearn
X = df_modelo.select([c for c in df_modelo.columns if c != "retrasado"]).to_numpy()
y = df_modelo["retrasado"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")

def evaluar_modelo(nombre, modelo):
    inicio = time.time()
    modelo.fit(X_train, y_train)
    t_entrenamiento = time.time() - inicio

    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cm  = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*50}\n  {nombre}\n{'='*50}")
    print(f"  Tiempo de entrenamiento: {t_entrenamiento:.2f} s")
    print(f"  Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")
    print(f"  Matriz de confusión:  VP={cm[1,1]:,} FN={cm[1,0]:,} | FP={cm[0,1]:,} VN={cm[0,0]:,}")

    return {"modelo": nombre, "tiempo": t_entrenamiento,
            "accuracy": acc, "f1": f1, "auc": auc, "cm": cm}

resultado_rl  = evaluar_modelo("Regresión Logística",
                               LogisticRegression(max_iter=1000, random_state=42))
resultado_rf  = evaluar_modelo("Random Forest",
                               RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
resultado_xgb = evaluar_modelo("XGBoost",
                               xgb.XGBClassifier(n_estimators=100, random_state=42,
                                                 n_jobs=-1, eval_metric="logloss", verbosity=0))

resultados_ml = [resultado_rl, resultado_rf, resultado_xgb]

print("\n\n=== RESUMEN COMPARATIVO ===")
print(f"{'Modelo':<22}{'Accuracy':>10}{'F1':>10}{'AUC':>10}{'Tiempo(s)':>12}")
print("-" * 64)
for r in resultados_ml:
    print(f"{r['modelo']:<22}{r['accuracy']:>10.4f}{r['f1']:>10.4f}{r['auc']:>10.4f}{r['tiempo']:>12.2f}")

### 6.2 Comparación de modelos (gráfica)

In [ ]:
nombres   = [r["modelo"].replace(" ", "\n") for r in resultados_ml]
acc_vals  = [r["accuracy"] for r in resultados_ml]
f1_vals   = [r["f1"] for r in resultados_ml]
auc_vals  = [r["auc"] for r in resultados_ml]

x = np.arange(len(nombres)); ancho = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - ancho, acc_vals, ancho, label="Accuracy", color="#3498db", edgecolor="white")
ax.bar(x,         f1_vals,  ancho, label="F1 Score", color="#2ecc71", edgecolor="white")
ax.bar(x + ancho, auc_vals, ancho, label="AUC",      color="#e74c3c", edgecolor="white")

ax.set_title("Comparación de Modelos de Machine Learning", fontsize=13, fontweight="bold")
ax.set_ylabel("Métrica")
ax.set_ylim(min(acc_vals + f1_vals + auc_vals) - 0.05, max(acc_vals + f1_vals + auc_vals) + 0.05)
ax.set_xticks(x); ax.set_xticklabels(nombres); ax.legend()

for i in range(len(nombres)):
    ax.text(i - ancho, acc_vals[i] + 0.002, f"{acc_vals[i]:.3f}", ha="center", fontsize=8)
    ax.text(i,         f1_vals[i]  + 0.002, f"{f1_vals[i]:.3f}",  ha="center", fontsize=8)
    ax.text(i + ancho, auc_vals[i] + 0.002, f"{auc_vals[i]:.3f}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("resultados_modelos.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: resultados_modelos.png")

## 7. Benchmark Polars vs Pandas

In [ ]:
import pandas as pd
import psutil

print("=== Información del sistema ===")
print(f"Núcleos disponibles:  {os.cpu_count()}")
print(f"RAM total:            {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"RAM disponible:       {psutil.virtual_memory().available / (1024**3):.1f} GB")
print(f"Tamaño del dataset:   {df_limpio.shape[0]:,} filas x {df_limpio.shape[1]} columnas")

tiempos_polars = {}
tiempos_pandas = {}

# 1) Lectura
t = time.time(); df_polars_bench = pl.read_csv(ruta_csv); tiempos_polars["lectura"] = time.time() - t
t = time.time(); df_pandas_bench = pd.read_csv(ruta_csv); tiempos_pandas["lectura"] = time.time() - t

# 2) Filtrado
t = time.time(); df_polars_filtrado = df_polars_bench.filter(pl.col("STATUS") == "ATA"); tiempos_polars["filtrado"] = time.time() - t
t = time.time(); df_pandas_filtrado = df_pandas_bench[df_pandas_bench["STATUS"] == "ATA"]; tiempos_pandas["filtrado"] = time.time() - t

# 3) Agregación
t = time.time()
agg_polars = df_polars_filtrado.group_by("DEPSTN").agg(pl.col("target").mean().alias("retraso_promedio"))
tiempos_polars["agregacion"] = time.time() - t
t = time.time()
agg_pandas = (df_pandas_filtrado.groupby("DEPSTN")["target"].mean()
              .reset_index().rename(columns={"target": "retraso_promedio"}))
tiempos_pandas["agregacion"] = time.time() - t

# 4) Join
t = time.time(); df_polars_join = df_polars_filtrado.join(agg_polars, on="DEPSTN", how="left"); tiempos_polars["join"] = time.time() - t
t = time.time(); df_pandas_join = df_pandas_filtrado.merge(agg_pandas, on="DEPSTN", how="left"); tiempos_pandas["join"] = time.time() - t

# 5) Feature engineering
t = time.time()
df_polars_fe = df_polars_join.with_columns([
    pl.col("DATOP").str.to_date(format="%Y-%m-%d").dt.month().alias("mes"),
    pl.col("DATOP").str.to_date(format="%Y-%m-%d").dt.weekday().alias("dia_semana"),
    pl.col("FLTID").str.slice(0, 2).str.strip_chars().alias("codigo_aerolinea"),
    (pl.col("target") > 15).cast(pl.Int8).alias("retrasado"),
])
tiempos_polars["feature_engineering"] = time.time() - t
t = time.time()
df_pandas_fe = df_pandas_join.copy()
df_pandas_fe["mes"] = pd.to_datetime(df_pandas_fe["DATOP"]).dt.month
df_pandas_fe["dia_semana"] = pd.to_datetime(df_pandas_fe["DATOP"]).dt.weekday
df_pandas_fe["codigo_aerolinea"] = df_pandas_fe["FLTID"].str[:2].str.strip()
df_pandas_fe["retrasado"] = (df_pandas_fe["target"] > 15).astype(int)
tiempos_pandas["feature_engineering"] = time.time() - t

# Total
tiempos_polars["total"] = sum(tiempos_polars.values())
tiempos_pandas["total"] = sum(tiempos_pandas.values())

print("\n=== TABLA RESUMEN BENCHMARK ===")
print(f"{'Operación':<22}{'Polars(s)':>12}{'Pandas(s)':>12}{'Speedup':>10}")
print("-" * 56)
for op in ["lectura", "filtrado", "agregacion", "join", "feature_engineering", "total"]:
    sp = tiempos_pandas[op] / tiempos_polars[op]
    print(f"{op:<22}{tiempos_polars[op]:>12.4f}{tiempos_pandas[op]:>12.4f}{sp:>9.2f}x")

### 7.1 Tiempos por operación (gráfica)

In [ ]:
ops    = ["lectura", "filtrado", "agregacion", "join", "feature_engineering", "total"]
labels = ["Lectura", "Filtrado", "Agregación", "Join", "Feature Eng.", "Total"]
t_pol  = [tiempos_polars[o] for o in ops]
t_pan  = [tiempos_pandas[o] for o in ops]

x = np.arange(len(labels)); ancho = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - ancho/2, t_pol, ancho, label="Polars", color="#3498db", edgecolor="white")
b2 = ax.bar(x + ancho/2, t_pan, ancho, label="Pandas", color="#e67e22", edgecolor="white")
ax.set_title("Benchmark Polars vs Pandas — Tiempo por Operación", fontsize=13, fontweight="bold")
ax.set_ylabel("Tiempo (segundos)")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.legend()
for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width()/2, b.get_height(),
                f"{b.get_height():.3f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("benchmark_tiempos.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: benchmark_tiempos.png")

### 7.2 Speedup por operación (gráfica)

Speedup = tiempo Pandas / tiempo Polars.

In [ ]:
speedups = [t_pan[i] / t_pol[i] for i in range(len(ops))]
colores  = ["#2ecc71" if s >= 1 else "#e74c3c" for s in speedups]

fig, ax = plt.subplots(figsize=(11, 5))
barras = ax.bar(labels, speedups, color=colores, edgecolor="white", linewidth=1.5)
ax.axhline(y=1, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Speedup de Polars sobre Pandas por Operación", fontsize=13, fontweight="bold")
ax.set_ylabel("Speedup (veces más rápido)")
for b, s in zip(barras, speedups):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
            f"{s:.2f}x", ha="center", fontweight="bold")
verde = mpatches.Patch(color="#2ecc71", label="Polars más rápido")
rojo  = mpatches.Patch(color="#e74c3c", label="Pandas más rápido")
ax.legend(handles=[verde, rojo])
plt.tight_layout()
plt.savefig("speedup_operaciones.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: speedup_operaciones.png")

## 8. Experimentos

### 8.1 Escalabilidad con tamaño de datos

In [ ]:
proporciones = [0.25, 0.50, 0.75, 1.00]
resultados_escalabilidad = []

for prop in proporciones:
    n = int(len(df_pandas_bench) * prop)
    sub_pl = df_polars_bench.head(n)
    sub_pd = df_pandas_bench.head(n)

    # Pipeline Polars
    ini = time.time()
    tmp = sub_pl.filter(pl.col("STATUS") == "ATA")
    agg = tmp.group_by("DEPSTN").agg(pl.col("target").mean().alias("retraso_promedio"))
    tmp = tmp.join(agg, on="DEPSTN", how="left")
    tmp = tmp.with_columns([
        pl.col("DATOP").str.to_date(format="%Y-%m-%d").dt.month().alias("mes"),
        pl.col("FLTID").str.slice(0, 2).str.strip_chars().alias("codigo_aerolinea"),
        (pl.col("target") > 15).cast(pl.Int8).alias("retrasado"),
    ])
    t_pl = time.time() - ini

    # Pipeline Pandas
    ini = time.time()
    tpd = sub_pd[sub_pd["STATUS"] == "ATA"]
    apd = (tpd.groupby("DEPSTN")["target"].mean()
           .reset_index().rename(columns={"target": "retraso_promedio"}))
    tpd = tpd.merge(apd, on="DEPSTN", how="left").copy()
    tpd["mes"] = pd.to_datetime(tpd["DATOP"]).dt.month
    tpd["codigo_aerolinea"] = tpd["FLTID"].str[:2].str.strip()
    tpd["retrasado"] = (tpd["target"] > 15).astype(int)
    t_pd = time.time() - ini

    resultados_escalabilidad.append({
        "proporcion": f"{int(prop*100)}%", "n_filas": n,
        "tiempo_polars": t_pl, "tiempo_pandas": t_pd, "speedup": t_pd / t_pl
    })
    print(f"{int(prop*100):>3}% ({n:>7,} filas) -> Polars {t_pl:.4f}s | "
          f"Pandas {t_pd:.4f}s | Speedup {t_pd/t_pl:.2f}x")

In [ ]:
labels_e = [r["proporcion"] for r in resultados_escalabilidad]
t_pol_e  = [r["tiempo_polars"] for r in resultados_escalabilidad]
t_pan_e  = [r["tiempo_pandas"] for r in resultados_escalabilidad]
sp_e     = [r["speedup"] for r in resultados_escalabilidad]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(labels_e, t_pol_e, marker="o", color="#3498db", linewidth=2, markersize=8, label="Polars")
axes[0].plot(labels_e, t_pan_e, marker="s", color="#e67e22", linewidth=2, markersize=8, label="Pandas")
axes[0].set_title("Tiempo Total vs Tamaño del Dataset")
axes[0].set_xlabel("Proporción del dataset"); axes[0].set_ylabel("Tiempo (s)"); axes[0].legend()

axes[1].plot(labels_e, sp_e, marker="D", color="#9b59b6", linewidth=2, markersize=8)
axes[1].axhline(y=1, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Speedup vs Tamaño del Dataset")
axes[1].set_xlabel("Proporción del dataset"); axes[1].set_ylabel("Speedup")
for i, s in enumerate(sp_e):
    axes[1].annotate(f"{s:.2f}x", (labels_e[i], s),
                     textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)

plt.suptitle("Experimento de Escalabilidad — Polars vs Pandas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("escalabilidad.png", bbox_inches="tight")
plt.show()
print("Gráfica guardada: escalabilidad.png")

### 8.2 Lazy Execution

In [ ]:
proceso = psutil.Process(os.getpid())

mem0 = proceso.memory_info().rss / (1024**2)
ini = time.time()
df_eager = (
    pl.read_csv(ruta_csv)
    .filter(pl.col("STATUS") == "ATA")
    .group_by("DEPSTN")
    .agg(pl.col("target").mean().alias("retraso_promedio"))
)
t_eager = time.time() - ini
mem_eager = proceso.memory_info().rss / (1024**2) - mem0

mem0 = proceso.memory_info().rss / (1024**2)
ini = time.time()
df_lazy = (
    pl.scan_csv(ruta_csv)
    .filter(pl.col("STATUS") == "ATA")
    .group_by("DEPSTN")
    .agg(pl.col("target").mean().alias("retraso_promedio"))
    .collect()
)
t_lazy = time.time() - ini
mem_lazy = proceso.memory_info().rss / (1024**2) - mem0

print("=== COMPARACIÓN EAGER vs LAZY ===")
print(f"{'Métrica':<18}{'Eager':>12}{'Lazy':>12}")
print("-" * 42)
print(f"{'Tiempo (s)':<18}{t_eager:>12.4f}{t_lazy:>12.4f}")
print(f"{'Memoria (MB)':<18}{mem_eager:>12.2f}{mem_lazy:>12.2f}")